# 05_pipeline_automation.ipynb

Automated pipeline: load processed data, construct multimodal feature sets, train models (binary & multi-class), save artifacts and metrics.

Requirements:
- src/ (data_processing.py, nlp_features.py, model_training.py, utils.py) present in PYTHONPATH
- config/config.yaml with `paths.processed`, `paths.artifacts`, `paths.models` and `nlp.max_features`, `nlp.svd_components`


# Step 1: Setup and imports

In [5]:
# Setup and imports
import os
import logging
import joblib
import pandas as pd
import numpy as np
import scipy.sparse as sp

# ML
from sklearn.decomposition import TruncatedSVD

# Project helpers
from src.utils import load_config, setup_logging, safe_join
from src.data_processing import load_processed, prepare_targets, prepare_features, split_for_tasks, save_pickle
from src.nlp_features import compute_simple_nlp_scores, compute_tfidf, save_vectorizer, transform_with_vectorizer
from src.model_training import ModelTrainer

# NB: If `src` not found, add project root to path
import sys
project_root = os.path.abspath(os.getcwd())
if project_root not in sys.path:
    sys.path.append(project_root)

# Config path (adjust if necessary)
cfg_path = "config/config.yaml"

# Load config and logger
cfg = load_config(cfg_path)
logger = setup_logging()
logger.info("Loaded config from %s", cfg_path)

# Create artifact folders (ensured by load_config, re-check)
artifacts_dir = cfg['paths']['artifacts']
models_dir = cfg['paths']['models']
processed_path = cfg['paths']['processed']
os.makedirs(artifacts_dir, exist_ok=True)
os.makedirs(models_dir, exist_ok=True)
logger.info("Artifacts dir: %s, models dir: %s", artifacts_dir, models_dir)


2025-10-15 20:05:06,335 INFO Loaded config from config/config.yaml
2025-10-15 20:05:06,342 INFO Artifacts dir: data/processed/model_artifacts, models dir: data/processed/model_artifacts/trained_models


In [8]:

from pathlib import Path
import os
import pandas as pd
import logging

logger = logging.getLogger()  # use existing logger from Cell 1 if run

# Absolute path to your processed folder (as you reported)
proc_dir = Path(r"C:\Users\AMAN PARGANIHA\AMAN PARGANIHA Dropbox\aman parganiha\My PC (LAPTOP-9RKITUJ5)\Desktop\project\data\processed")

if not proc_dir.exists():
    raise FileNotFoundError(f"Directory not found: {proc_dir}")

# look for likely CSV files
candidates = sorted(proc_dir.glob("credit_ratings*.csv")) + sorted(proc_dir.glob("*.csv"))
candidates = list(dict.fromkeys(candidates))  # unique, keep order

if not candidates:
    raise FileNotFoundError(f"No CSV files found in {proc_dir}. Place your final CSV there or update config to point to the file.")

print("Found CSV files:")
for i, p in enumerate(candidates):
    try:
        size = p.stat().st_size
    except Exception:
        size = "unknown"
    print(f" [{i}] {p.name} — {size} bytes")

# auto-select: prefer a file matching credit_ratings*, else take first CSV
selected = None
for p in candidates:
    if p.name.lower().startswith("credit_ratings"):
        selected = p
        break
if selected is None:
    selected = candidates[0]

print("\nAuto-selected file:", selected)
# permission check
if not os.access(selected, os.R_OK):
    raise PermissionError(
        f"Permission denied for file: {selected}\n"
        "If the file is in Dropbox, try pausing sync or copying the file to a local folder (e.g., Desktop) and updating config."
    )

# attempt to load
print("Loading CSV (this may take a moment)...")
df_test = pd.read_csv(selected, nrows=5)  # quick sanity read first
print("Quick preview loaded (first 5 rows):")
display(df_test)

# If all good, set cfg['paths']['processed'] to this file path for later cells
try:
    cfg['paths']['processed'] = str(selected)
    print("Updated cfg['paths']['processed'] to:", cfg['paths']['processed'])
except Exception:
    print("Note: cfg not in scope (make sure you run Cell 1 first). If cfg exists, it will be updated.")


Found CSV files:
 [0] credit_ratings_86k.csv — 4769458 bytes
 [1] credit_ratings_cleaned.csv — 12442629 bytes
 [2] credit_ratings_multimodal_86k.csv — 17665317 bytes
 [3] credit_ratings_multimodal_final.csv — 18869480 bytes
 [4] feature_importance.csv — 972 bytes
 [5] nlp_features.csv — 7283590 bytes
 [6] sample_10k_companies.csv — 2256641 bytes
 [7] sec_financial_data_86k.csv — 10790558 bytes

Auto-selected file: C:\Users\AMAN PARGANIHA\AMAN PARGANIHA Dropbox\aman parganiha\My PC (LAPTOP-9RKITUJ5)\Desktop\project\data\processed\credit_ratings_86k.csv
Loading CSV (this may take a moment)...
Quick preview loaded (first 5 rows):


,adsh,company_name,sector,rating,investment_grade,financial_score
0,0000002178-22-000033,Company_1,Technology,BBB,1,2.00
1,0000002178-22-000046,Company_2,Financial,BB,0,1.01
2,0000002178-22-000066,Company_3,Healthcare,BB,0,1.22
3,0000002178-22-000089,Company_4,Energy,AA+,1,4.94
4,0000002178-23-000038,Company_5,Consumer,BBB,1,1.68


Updated cfg['paths']['processed'] to: C:\Users\AMAN PARGANIHA\AMAN PARGANIHA Dropbox\aman parganiha\My PC (LAPTOP-9RKITUJ5)\Desktop\project\data\processed\credit_ratings_86k.csv


# --- Switch cfg to use credit_ratings_multimodal_final.csv ---

In [10]:
# --- Switch cfg to use credit_ratings_multimodal_final.csv ---
from pathlib import Path
proc_dir = Path(r"C:\Users\AMAN PARGANIHA\AMAN PARGANIHA Dropbox\aman parganiha\My PC (LAPTOP-9RKITUJ5)\Desktop\project\data\processed")
candidate = proc_dir / "credit_ratings_multimodal_final.csv"
if not candidate.exists():
    raise FileNotFoundError(f"Expected file not found: {candidate}\nRun the listing cell again to confirm exact filename.")
cfg['paths']['processed'] = str(candidate)
print("cfg['paths']['processed'] updated to:", cfg['paths']['processed'])
# Now run the combined preparation cell 
# --- Continue with currently selected processed file (credit_ratings_86k.csv) ---
# This cell prepares targets, features and basic NLP scores (replacement for remainder of Cell 2).

import pandas as pd
import numpy as np
from pathlib import Path
from src.data_processing import prepare_targets, prepare_features
from src.nlp_features import compute_simple_nlp_scores

# cfg already updated by the discovery cell earlier
proc_fp = Path(cfg['paths']['processed'])
print("Using processed CSV:", proc_fp)

# Full load (now that quick-read passed, read fully)
df = pd.read_csv(proc_fp)
print("Full dataset loaded:", df.shape)

# Targets
y_binary, y_multi = prepare_targets(df,
                                   rating_col=cfg.get('column_names', {}).get('rating','rating'),
                                   inv_grade_col=cfg.get('column_names', {}).get('investment_grade','investment_grade'))
print("Targets prepared: binary (investment_grade) and multi (rating).")

# Base tabular features (excludes reasonable default columns)
X_base, base_features = prepare_features(df, exclude_cols=cfg.get('exclude_cols', None))
print(f"Base features detected: {len(base_features)} columns (showing up to 10):", base_features[:10])

# MD&A detection (try common column names)
mdna_col = cfg.get('column_names', {}).get('mda_text', 'mda_text')
if mdna_col not in df.columns:
    for alt in ['mda_text','mdna_text','mda','mdna','management_discussion', 'management_discussion_and_analysis','md&a','mdna_text']:
        if alt in df.columns:
            mdna_col = alt
            print("Auto-detected MD&A column:", mdna_col)
            break

if mdna_col not in df.columns:
    print("MD&A column not found; text-based scenarios will use empty strings. If your multimodal file has MD&A, consider switching to it.")
    mdna_series = pd.Series([''] * len(df))
else:
    mdna_series = df[mdna_col].fillna('')

# Compute simple NLP scores (token count, avg word len)
nlp_scores = compute_simple_nlp_scores(mdna_series)
print("Computed NLP scores shape:", nlp_scores.shape, "Columns:", list(nlp_scores.columns))

# Save small quick artifacts for downstream cells
# (these variables will be referenced by subsequent cells)
globals().update({
    'df': df,
    'y_binary': y_binary,
    'y_multi': y_multi,
    'X_base': X_base,
    'base_features': base_features,
    'mdna_col': mdna_col,
    'mdna_series': mdna_series,
    'nlp_scores': nlp_scores,
})
print("Preparation complete — proceed to Cell 3 (splits) and then continue through the notebook.")



cfg['paths']['processed'] updated to: C:\Users\AMAN PARGANIHA\AMAN PARGANIHA Dropbox\aman parganiha\My PC (LAPTOP-9RKITUJ5)\Desktop\project\data\processed\credit_ratings_multimodal_final.csv
Using processed CSV: C:\Users\AMAN PARGANIHA\AMAN PARGANIHA Dropbox\aman parganiha\My PC (LAPTOP-9RKITUJ5)\Desktop\project\data\processed\credit_ratings_multimodal_final.csv
Full dataset loaded: (35098, 47)
Targets prepared: binary (investment_grade) and multi (rating).
Base features detected: 42 columns (showing up to 10): ['sector', 'accounts_receivable', 'cash', 'current_assets', 'current_liabilities', 'gross_profit', 'inventory', 'long_term_debt', 'net_income', 'operating_income']
MD&A column not found; text-based scenarios will use empty strings. If your multimodal file has MD&A, consider switching to it.
Computed NLP scores shape: (35098, 2) Columns: ['tok_count', 'avg_word_len']
Preparation complete — proceed to Cell 3 (splits) and then continue through the notebook.


# Step 2: - Train/test splits for both tasks (we will re-use same X for various feature sets)

In [11]:
# Cell 3 - Train/test splits for both tasks (we will re-use same X for various feature sets)
split_cfg = cfg.get('split', {})
test_size = split_cfg.get('test_size', 0.2)
random_state = split_cfg.get('random_state', 42)

X_train_b, X_test_b, y_train_b, y_test_b, X_train_m, X_test_m, y_train_m, y_test_m = \
    split_for_tasks(X_base.join(nlp_scores), y_binary, y_multi, test_size=test_size, random_state=random_state)

logger.info("Binary train/test sizes: %d / %d", len(y_train_b), len(y_test_b))
logger.info("Multi-class train/test sizes: %d / %d", len(y_train_m), len(y_test_m))


2025-10-15 20:21:08,962 INFO Binary train/test sizes: 28078 / 7020
2025-10-15 20:21:08,962 INFO Multi-class train/test sizes: 28078 / 7020


# ===== Robust TF-IDF + SVD with graceful fallback =====

In [14]:
# ===== Robust TF-IDF + SVD with graceful fallback =====
import numpy as np
import joblib
from pathlib import Path
from sklearn.decomposition import TruncatedSVD
from src.nlp_features import compute_tfidf, save_vectorizer

# Assumptions: mdna_series, tabular_nlp_df, cfg, models_dir are present (from previous cells)
assert 'mdna_series' in globals(), "mdna_series missing — run preparation cell first."
assert 'tabular_nlp_df' in globals() or 'tabular_nlp_arr' in globals(), "tabular_nlp_df/arr missing."

n = len(mdna_series)
logger.info("Robust TF-IDF: checking MD&A corpus (n=%d)", n)

# Count non-empty docs
non_empty_mask = mdna_series.fillna('').astype(str).map(lambda s: bool(s.strip()))
n_nonempty = non_empty_mask.sum()
logger.info("Non-empty MD&A docs: %d / %d", int(n_nonempty), int(n))

# Configurable thresholds
min_docs_for_tfidf = max(10, int(0.001 * n))  # at least 10 docs or 0.1% of corpus
max_features = cfg.get('nlp', {}).get('max_features', 5000)
n_svd = cfg.get('nlp', {}).get('svd_components', 200)

use_tfidf = n_nonempty >= min_docs_for_tfidf

tfidf_path = os.path.join(models_dir, 'tfidf_vectorizer.joblib')
svd_path = os.path.join(models_dir, 'tfidf_svd.joblib')
fallback_marker = os.path.join(models_dir, 'tfidf_fallback.txt')

if use_tfidf:
    try:
        corpus_nonempty = mdna_series.fillna('').astype(str).tolist()
        logger.info("Attempting TF-IDF (max_features=%d) on MD&A corpus", max_features)
        vec, X_tfidf = compute_tfidf(corpus_nonempty, max_features=max_features, ngram_range=(1,2))
        save_vectorizer(vec, tfidf_path)
        logger.info("TF-IDF vectorizer saved to %s", tfidf_path)

        n_components = min(n_svd, max(1, X_tfidf.shape[1] - 1))
        svd = TruncatedSVD(n_components=n_components, random_state=42)
        X_svd = svd.fit_transform(X_tfidf)
        joblib.dump(svd, svd_path)
        logger.info("SVD saved to %s; X_svd shape: %s", svd_path, X_svd.shape)

        # remove any previous fallback marker
        if os.path.exists(fallback_marker):
            os.remove(fallback_marker)

    except ValueError as e:
        # Typical message: "empty vocabulary; perhaps the documents only contain stop words"
        logger.warning("TF-IDF failed with ValueError: %s", e)
        logger.warning("Falling back to zero SVD features (no text signal).")
        X_svd = np.zeros((n, min(1, n_svd)))  # at least one zero column
        # write fallback marker for traceability
        with open(fallback_marker, 'w', encoding='utf-8') as f:
            f.write(f"TF-IDF failed: {repr(e)}\nFalling back to zeros. n_nonempty={n_nonempty}\n")
        # ensure no stale vectorizer/svd files left
        if os.path.exists(tfidf_path): os.remove(tfidf_path)
        if os.path.exists(svd_path): os.remove(svd_path)

else:
    logger.warning("Only %d non-empty MD&A docs (<%d threshold). Skipping TF-IDF and using zero SVD features.",
                   int(n_nonempty), min_docs_for_tfidf)
    X_svd = np.zeros((n, min(1, n_svd)))
    with open(fallback_marker, 'w', encoding='utf-8') as f:
        f.write(f"Insufficient non-empty MD&A docs: n_nonempty={n_nonempty}, threshold={min_docs_for_tfidf}\n")

# At this point X_svd is a dense matrix (n, k). If k==1 it's a single zero column in fallback.
logger.info("X_svd final shape: %s", X_svd.shape)

# Now create or update feature_sets['tabular_fulltext'] by concatenating tabular_nlp_df numeric + X_svd
# Ensure tabular_nlp_df variable exists; otherwise create from X_base + nlp_scores
if 'tabular_nlp_df' not in globals():
    # try to build it from available pieces
    X_base_num = X_base.select_dtypes(include=[np.number]).reset_index(drop=True)
    tabular_nlp_df = pd.concat([X_base_num, nlp_scores.reset_index(drop=True)], axis=1).select_dtypes(include=[np.number])

tab_nlp_numeric = tabular_nlp_df.values  # shape (n, p)
X_full_dense = np.hstack([tab_nlp_numeric, X_svd])

# Update feature_sets (if previous variable exists)
if 'feature_sets' in globals():
    feature_sets['tabular_fulltext'] = (X_full_dense, list(tabular_nlp_df.columns) + [f"svd_{i}" for i in range(X_svd.shape[1])])
else:
    feature_sets = {
        'tabular': (X_base.select_dtypes(include=[np.number]).values, list(X_base.select_dtypes(include=[np.number]).columns)),
        'tabular_nlp': (tab_nlp_numeric, list(tabular_nlp_df.columns)),
        'tabular_fulltext': (X_full_dense, list(tabular_nlp_df.columns) + [f"svd_{i}" for i in range(X_svd.shape[1])])
    }

# Save a tiny summary file for traceability
summary_info = {
    'n_docs': int(n),
    'n_nonempty': int(n_nonempty),
    'used_tfidf': bool(use_tfidf and 'vec' in locals()),
    'tfidf_path': tfidf_path if os.path.exists(tfidf_path) else None,
    'svd_path': svd_path if os.path.exists(svd_path) else None,
    'fallback_marker': fallback_marker if os.path.exists(fallback_marker) else None
}
joblib.dump(summary_info, os.path.join(models_dir, 'tfidf_run_summary.joblib'))
logger.info("Saved TF-IDF run summary: %s", summary_info)

print("TF-IDF step completed. used_tfidf =", summary_info['used_tfidf'],
      "; non-empty docs =", summary_info['n_nonempty'], "/", summary_info['n_docs'])
if not summary_info['used_tfidf']:
    print("→ Notice: text features are not available (fallback used). For real text features, run using the multimodal CSV that contains MD&A text (e.g. credit_ratings_multimodal_final.csv).")


2025-10-15 20:34:30,091 INFO Robust TF-IDF: checking MD&A corpus (n=35098)
2025-10-15 20:34:30,128 INFO Non-empty MD&A docs: 0 / 35098
2025-10-15 20:34:30,132 WARNING Only 0 non-empty MD&A docs (<35 threshold). Skipping TF-IDF and using zero SVD features.
2025-10-15 20:34:30,136 INFO X_svd final shape: (35098, 1)
2025-10-15 20:34:30,179 INFO Saved TF-IDF run summary: {'n_docs': 35098, 'n_nonempty': 0, 'used_tfidf': False, 'tfidf_path': None, 'svd_path': None, 'fallback_marker': 'data/processed/model_artifacts/trained_models\\tfidf_fallback.txt'}


TF-IDF step completed. used_tfidf = False ; non-empty docs = 0 / 35098
→ Notice: text features are not available (fallback used). For real text features, run using the multimodal CSV that contains MD&A text (e.g. credit_ratings_multimodal_final.csv).


# ===== Option A: Switch to multimodal CSV and recompute text features =====

In [15]:
# ===== Option A: Switch to multimodal CSV and recompute text features =====
from pathlib import Path
import os
import joblib
import numpy as np
import pandas as pd

# 1) Update cfg to point to the multimodal file
proc_dir = Path(r"C:\Users\AMAN PARGANIHA\AMAN PARGANIHA Dropbox\aman parganiha\My PC (LAPTOP-9RKITUJ5)\Desktop\project\data\processed")
multimodal_fp = proc_dir / "credit_ratings_multimodal_final.csv"
if not multimodal_fp.exists():
    raise FileNotFoundError(f"Multimodal file not found: {multimodal_fp}\nCheck the filename in the folder listing.")
cfg['paths']['processed'] = str(multimodal_fp)
print("cfg['paths']['processed'] set to:", cfg['paths']['processed'])

# 2) Reload dataset fully and prepare targets & features (same as earlier prep cell)
df = pd.read_csv(cfg['paths']['processed'])
print("Loaded multimodal CSV shape:", df.shape)

from src.data_processing import prepare_targets, prepare_features
from src.nlp_features import compute_simple_nlp_scores

y_binary, y_multi = prepare_targets(df,
                                   rating_col=cfg.get('column_names', {}).get('rating','rating'),
                                   inv_grade_col=cfg.get('column_names', {}).get('investment_grade','investment_grade'))

X_base, base_features = prepare_features(df, exclude_cols=cfg.get('exclude_cols', None))
print("Base numeric features:", len(base_features))

# 3) Detect MD&A column (try common names) and build mdna_series
mdna_col = cfg.get('column_names', {}).get('mda_text', 'mda_text')
if mdna_col not in df.columns:
    for alt in ['mda_text','mdna_text','mda','mdna','management_discussion','management_discussion_and_analysis','md&a']:
        if alt in df.columns:
            mdna_col = alt
            break

if mdna_col not in df.columns:
    print("WARNING: MD&A column still not found in multimodal file. Aborting switch.")
else:
    print("Using MD&A column:", mdna_col)
    mdna_series = df[mdna_col].fillna('').astype(str)

    # 4) Recompute simple NLP numeric scores and overwrite nlp_scores
    nlp_scores = compute_simple_nlp_scores(mdna_series)
    print("Recomputed nlp_scores shape:", nlp_scores.shape)

    # 5) (Optional) Lower TF-IDF dimensions if you worry about memory (uncomment & set)
    # cfg['nlp']['max_features'] = 2000
    # cfg['nlp']['svd_components'] = 150

    # 6) Re-run the robust TF-IDF + SVD cell (paste the robust TF-IDF cell below here)
    # --- Begin Robust TF-IDF + SVD block (copy from the cell I provided earlier) ---
    import logging
    from sklearn.decomposition import TruncatedSVD
    from src.nlp_features import compute_tfidf, save_vectorizer
    logger = logging.getLogger()

    n = len(mdna_series)
    non_empty_mask = mdna_series.fillna('').map(lambda s: bool(s.strip()))
    n_nonempty = non_empty_mask.sum()
    min_docs_for_tfidf = max(10, int(0.001 * n))
    max_features = cfg.get('nlp', {}).get('max_features', 5000)
    n_svd = cfg.get('nlp', {}).get('svd_components', 200)

    models_dir = cfg['paths'].get('models', os.path.join(cfg['paths']['artifacts'], 'trained_models'))
    os.makedirs(models_dir, exist_ok=True)
    tfidf_path = os.path.join(models_dir, 'tfidf_vectorizer.joblib')
    svd_path = os.path.join(models_dir, 'tfidf_svd.joblib')
    fallback_marker = os.path.join(models_dir, 'tfidf_fallback.txt')

    if n_nonempty >= min_docs_for_tfidf:
        try:
            vec, X_tfidf = compute_tfidf(mdna_series.tolist(), max_features=max_features, ngram_range=(1,2))
            save_vectorizer(vec, tfidf_path)
            n_components = min(n_svd, max(1, X_tfidf.shape[1]-1))
            svd = TruncatedSVD(n_components=n_components, random_state=42)
            X_svd = svd.fit_transform(X_tfidf)
            joblib.dump(svd, svd_path)
            if os.path.exists(fallback_marker): os.remove(fallback_marker)
            print("TF-IDF + SVD succeeded. X_svd shape:", X_svd.shape)
        except Exception as e:
            print("TF-IDF failed:", e)
            X_svd = np.zeros((n, min(1, n_svd)))
            with open(fallback_marker, 'w', encoding='utf-8') as f:
                f.write(f"TF-IDF failed: {repr(e)}\n")
    else:
        print(f"Not enough non-empty docs ({n_nonempty}/{n}). Skipping TF-IDF.")
        X_svd = np.zeros((n, min(1, n_svd)))
        with open(fallback_marker, 'w', encoding='utf-8') as f:
            f.write(f"Insufficient non-empty MD&A docs: {n_nonempty}/{n}\n")

    # 7) Build feature_sets (dense) like before and save train/test indices
    X_base_num = X_base.select_dtypes(include=[np.number]).reset_index(drop=True)
    tabular_arr = X_base_num.values
    tabular_nlp_df = pd.concat([X_base_num, nlp_scores.reset_index(drop=True)], axis=1).select_dtypes(include=[np.number])
    tabular_nlp_arr = tabular_nlp_df.values
    X_full_dense = np.hstack([tabular_nlp_arr, X_svd])
    feature_sets = {
        'tabular': (tabular_arr, list(X_base_num.columns)),
        'tabular_nlp': (tabular_nlp_arr, list(tabular_nlp_df.columns)),
        'tabular_fulltext': (X_full_dense, list(tabular_nlp_df.columns) + [f"svd_{i}" for i in range(X_svd.shape[1])])
    }
    # Save indices for reproducibility
    from sklearn.model_selection import train_test_split
    n = len(df)
    indices = np.arange(n)
    test_size = cfg.get('split', {}).get('test_size', 0.2)
    random_state = cfg.get('split', {}).get('random_state', 42)
    idx_train_b, idx_test_b = train_test_split(indices, test_size=test_size, stratify=y_binary, random_state=random_state)[:2]
    idx_train_m, idx_test_m = train_test_split(indices, test_size=test_size, stratify=y_multi, random_state=random_state)[:2]
    np.save(os.path.join(models_dir, 'idx_train_b.npy'), np.sort(idx_train_b))
    np.save(os.path.join(models_dir, 'idx_test_b.npy'), np.sort(idx_test_b))
    np.save(os.path.join(models_dir, 'idx_train_m.npy'), np.sort(idx_train_m))
    np.save(os.path.join(models_dir, 'idx_test_m.npy'), np.sort(idx_test_m))
    print("Feature sets built and indices saved. Proceed to run the training cells (trainer.run).")


cfg['paths']['processed'] set to: C:\Users\AMAN PARGANIHA\AMAN PARGANIHA Dropbox\aman parganiha\My PC (LAPTOP-9RKITUJ5)\Desktop\project\data\processed\credit_ratings_multimodal_final.csv
Loaded multimodal CSV shape: (35098, 47)
Base numeric features: 42


# Auto-detect MD&A-like column, show top candidates, and (if found) recompute NLP scores + TF-IDF+SVD

In [16]:
# Auto-detect MD&A-like column, show top candidates, and (if found) recompute NLP scores + TF-IDF+SVD
import pandas as pd
import numpy as np
import os
import joblib
from pathlib import Path
from sklearn.decomposition import TruncatedSVD
from src.nlp_features import compute_simple_nlp_scores, compute_tfidf, save_vectorizer

# Preconditions
assert 'df' in globals(), "df not found — please run the previous cells that loaded the multimodal CSV."
models_dir = cfg['paths'].get('models', os.path.join(cfg['paths']['artifacts'], 'trained_models'))
os.makedirs(models_dir, exist_ok=True)

# 1) Identify object/text columns
obj_cols = df.select_dtypes(include=['object']).columns.tolist()
print(f"Found {len(obj_cols)} object columns. Scanning for text-like columns...")

# compute basic stats for each candidate
cand_stats = []
for c in obj_cols:
    ser = df[c].fillna('').astype(str)
    non_empty = ser.map(lambda s: bool(s.strip())).sum()
    avg_len = ser.map(len).mean()
    med_len = ser.map(len).median()
    cand_stats.append((c, int(non_empty), avg_len, med_len))

# create DataFrame sorted by avg_len
cand_df = pd.DataFrame(cand_stats, columns=['col','non_empty_count','avg_len','median_len'])
cand_df = cand_df.sort_values('avg_len', ascending=False).reset_index(drop=True)
display(cand_df.head(12))

# 2) Look for keyword matches in column names (prefer these)
keywords = ['md&a','mda','management','discussion','analysis','mdna','narrative','managment','disclosure','disclosures','managementdiscussion']
cand_df['keyword_match'] = cand_df['col'].str.lower().apply(lambda s: any(k in s for k in keywords))

# show top keyword-matched columns first
cand_df = cand_df.sort_values(['keyword_match','avg_len'], ascending=[False, False]).reset_index(drop=True)
print("\nTop candidate columns (keyword matches prioritized):")
display(cand_df.head(8))

# 3) Auto-select best candidate:
# Criteria: keyword match with non-empty > 0 and avg_len>20 OR highest avg_len with >0 non-empty
selected_col = None
# first try keyword match with reasonable content
for _, row in cand_df.iterrows():
    if row['keyword_match'] and row['non_empty_count'] > 10 and row['avg_len'] > 50:
        selected_col = row['col']
        break
# fallback: choose the column with max avg_len and some non-empty rows
if selected_col is None:
    for _, row in cand_df.iterrows():
        if row['non_empty_count'] > 10 and row['avg_len'] > 50:
            selected_col = row['col']
            break
# last resort: accept any column with non_empty_count > 0 and avg_len > 10
if selected_col is None:
    for _, row in cand_df.iterrows():
        if row['non_empty_count'] > 0 and row['avg_len'] > 10:
            selected_col = row['col']
            break

if selected_col is None:
    print("No suitable MD&A-like column auto-detected. The multimodal CSV may not contain full MD&A text under any column.")
    print("Options: (a) Inspect the above candidate table and tell me which column to use, or (b) re-run using a different CSV that contains the text (e.g. the original multimodal file you expect).")
else:
    print("Auto-selected MD&A-like column:", selected_col)
    # show sample values
    print("\nSample (first 6 non-empty entries):")
    sample_nonempty = df[selected_col].fillna('').astype(str).map(lambda s: s.strip()).replace('', np.nan).dropna()
    display(sample_nonempty.head(6))
    # build mdna_series and recompute nlp_scores
    mdna_series = df[selected_col].fillna('').astype(str).reset_index(drop=True)
    nlp_scores = compute_simple_nlp_scores(mdna_series)
    print("Recomputed simple NLP scores shape:", nlp_scores.shape, "; columns:", list(nlp_scores.columns))

    # 4) Robust TF-IDF + SVD (same fallback logic as before)
    n = len(mdna_series)
    non_empty_mask = mdna_series.map(lambda s: bool(s.strip()))
    n_nonempty = non_empty_mask.sum()
    min_docs_for_tfidf = max(10, int(0.001 * n))
    max_features = cfg.get('nlp', {}).get('max_features', 5000)
    n_svd = cfg.get('nlp', {}).get('svd_components', 200)

    tfidf_path = os.path.join(models_dir, 'tfidf_vectorizer.joblib')
    svd_path = os.path.join(models_dir, 'tfidf_svd.joblib')
    fallback_marker = os.path.join(models_dir, 'tfidf_fallback.txt')

    if n_nonempty >= min_docs_for_tfidf:
        try:
            print(f"\nRunning TF-IDF (max_features={max_features}) on {int(n_nonempty)} non-empty docs...")
            vec, X_tfidf = compute_tfidf(mdna_series.tolist(), max_features=max_features, ngram_range=(1,2))
            save_vectorizer(vec, tfidf_path)
            n_components = min(n_svd, max(1, X_tfidf.shape[1]-1))
            svd = TruncatedSVD(n_components=n_components, random_state=42)
            X_svd = svd.fit_transform(X_tfidf)
            joblib.dump(svd, svd_path)
            if os.path.exists(fallback_marker): os.remove(fallback_marker)
            print("TF-IDF + SVD succeeded. X_svd shape:", X_svd.shape)
            used_tfidf = True
        except Exception as e:
            print("TF-IDF failed with exception:", repr(e))
            print("Falling back to zero SVD features.")
            X_svd = np.zeros((n, min(1, n_svd)))
            with open(fallback_marker, 'w', encoding='utf-8') as f:
                f.write(f"TF-IDF failed: {repr(e)}\n")
            used_tfidf = False
    else:
        print(f"Only {int(n_nonempty)} non-empty docs (<{min_docs_for_tfidf} threshold). Skipping TF-IDF and using zero SVD features.")
        X_svd = np.zeros((n, min(1, n_svd)))
        with open(fallback_marker, 'w', encoding='utf-8') as f:
            f.write(f"Insufficient non-empty MD&A docs: n_nonempty={n_nonempty}, n={n}\n")
        used_tfidf = False

    # 5) Build feature_sets and save indices (same as before)
    X_base_num = X_base.select_dtypes(include=[np.number]).reset_index(drop=True)
    tabular_arr = X_base_num.values
    tabular_nlp_df = pd.concat([X_base_num, nlp_scores.reset_index(drop=True)], axis=1).select_dtypes(include=[np.number])
    tabular_nlp_arr = tabular_nlp_df.values
    X_full_dense = np.hstack([tabular_nlp_arr, X_svd])

    feature_sets = {
        'tabular': (tabular_arr, list(X_base_num.columns)),
        'tabular_nlp': (tabular_nlp_arr, list(tabular_nlp_df.columns)),
        'tabular_fulltext': (X_full_dense, list(tabular_nlp_df.columns) + [f"svd_{i}" for i in range(X_svd.shape[1])])
    }

    # Save train/test index arrays for reproducibility
    from sklearn.model_selection import train_test_split
    indices = np.arange(len(df))
    test_size = cfg.get('split', {}).get('test_size', 0.2)
    random_state = cfg.get('split', {}).get('random_state', 42)
    idx_train_b, idx_test_b = train_test_split(indices, test_size=test_size, stratify=y_binary, random_state=random_state)[:2]
    idx_train_m, idx_test_m = train_test_split(indices, test_size=test_size, stratify=y_multi, random_state=random_state)[:2]
    np.save(os.path.join(models_dir, 'idx_train_b.npy'), np.sort(idx_train_b))
    np.save(os.path.join(models_dir, 'idx_test_b.npy'), np.sort(idx_test_b))
    np.save(os.path.join(models_dir, 'idx_train_m.npy'), np.sort(idx_train_m))
    np.save(os.path.join(models_dir, 'idx_test_m.npy'), np.sort(idx_test_m))
    joblib.dump({
        'n_docs': int(n),
        'n_nonempty': int(n_nonempty),
        'used_tfidf': used_tfidf,
        'tfidf_path': tfidf_path if os.path.exists(tfidf_path) else None,
        'svd_path': svd_path if os.path.exists(svd_path) else None
    }, os.path.join(models_dir, 'tfidf_run_summary.joblib'))

    print("\nFeature sets rebuilt. Shapes:")
    print(" - tabular:", feature_sets['tabular'][0].shape)
    print(" - tabular_nlp:", feature_sets['tabular_nlp'][0].shape)
    print(" - tabular_fulltext:", feature_sets['tabular_fulltext'][0].shape)
    print("TF-IDF used:", used_tfidf)


Found 5 object columns. Scanning for text-like columns...


,col,non_empty_count,avg_len,median_len
0,adsh,35098,20.000000,20.0
1,company_name,35098,12.874437,13.0
2,sector,35098,8.852413,9.0
3,company_size,35098,5.788449,5.0
4,rating,35098,2.084706,2.0



Top candidate columns (keyword matches prioritized):


,col,non_empty_count,avg_len,median_len,keyword_match
0,adsh,35098,20.000000,20.0,False
1,company_name,35098,12.874437,13.0,False
2,sector,35098,8.852413,9.0,False
3,company_size,35098,5.788449,5.0,False
4,rating,35098,2.084706,2.0,False


Auto-selected MD&A-like column: adsh

Sample (first 6 non-empty entries):


0    0000002178-23-000038
1    0000002178-23-000082
2    0000002178-24-000035
3    0000002178-24-000076
4    0000002178-24-000096
5    0000002488-22-000123
Name: adsh, dtype: object

Recomputed simple NLP scores shape: (35098, 2) ; columns: ['tok_count', 'avg_word_len']

Running TF-IDF (max_features=5000) on 35098 non-empty docs...
TF-IDF + SVD succeeded. X_svd shape: (35098, 200)

Feature sets rebuilt. Shapes:
 - tabular: (35098, 40)
 - tabular_nlp: (35098, 42)
 - tabular_fulltext: (35098, 242)
TF-IDF used: True


# Robust helper: find true long-text column, show samples, allow forced selection, then rebuild TF-IDF+SVD + feature_sets

In [18]:
# Robust helper: find true long-text column, show samples, allow forced selection, then rebuild TF-IDF+SVD + feature_sets

import os, joblib, numpy as np, pandas as pd
from sklearn.decomposition import TruncatedSVD
from src.nlp_features import compute_tfidf, save_vectorizer, compute_simple_nlp_scores
from pathlib import Path
from collections import Counter

models_dir = cfg['paths'].get('models', os.path.join(cfg['paths']['artifacts'], 'trained_models'))
os.makedirs(models_dir, exist_ok=True)

# ✅ Set this manually if you already know the text column (None = auto-detect)
FORCE_SELECTED_COL = None  # e.g., "mdna_text" or "mda"

# Gather object columns and compute heuristics
obj_cols = df.select_dtypes(include=['object']).columns.tolist()
stats = []
for c in obj_cols:
    ser = df[c].fillna('').astype(str)
    n = len(ser)
    non_empty = ser.map(lambda s: bool(s.strip())).sum()
    avg_len = ser.map(len).mean()
    median_len = ser.map(len).median()
    tok_counts = ser.map(lambda s: len(s.split()))
    avg_tok = tok_counts.mean()
    ws_ratio = ser.map(lambda s: s.count(' ') / max(1, len(s))).mean()
    punct_ratio = ser.map(lambda s: sum(1 for ch in s if ch in '.,;:!?') / max(1, len(s))).mean()
    distinct_ratio = ser.nunique() / max(1, n)
    stats.append({
        'col': c,
        'non_empty': int(non_empty),
        'avg_len': float(avg_len),
        'median_len': float(median_len),
        'avg_tok': float(avg_tok),
        'ws_ratio': float(ws_ratio),
        'punct_ratio': float(punct_ratio),
        'distinct_ratio': float(distinct_ratio)
    })

stat_df = pd.DataFrame(stats).sort_values(['avg_tok','avg_len','punct_ratio'], ascending=False).reset_index(drop=True)
print("Candidate text-like columns (sorted by avg token count):")
display(stat_df.head(12))

# Show samples
top_k = 8
print("\nShowing up to 3 sample values for top text candidates:\n")
for i, row in stat_df.head(top_k).iterrows():
    col = row['col']
    ser = df[col].fillna('').astype(str).map(lambda s: s.strip())
    nonempty = ser[ser.str.len() > 0]
    sample = nonempty.head(3).tolist()
    print(f"--- {col}  | non_empty={row['non_empty']}, avg_tok={row['avg_tok']:.1f} ---")
    for j, s_val in enumerate(sample):
        safe_sample = s_val.replace('\n', ' ').replace('\r', ' ')
        print(f"  sample {j+1}: {safe_sample[:200]}")
    if len(sample) == 0:
        print("  (no non-empty samples)")
    print()

# Column selection logic
selected_col = None
if FORCE_SELECTED_COL:
    if FORCE_SELECTED_COL in df.columns:
        selected_col = FORCE_SELECTED_COL
        print("FORCED selection:", selected_col)
    else:
        raise ValueError(f"FORCE_SELECTED_COL='{FORCE_SELECTED_COL}' not found.")
else:
    for _, r in stat_df.iterrows():
        if (r['avg_tok'] > 30 and r['non_empty'] > 50) or (r['avg_len'] > 200):
            selected_col = r['col']
            break

if not selected_col:
    print("\n⚠ No strong text column found automatically — pick one manually from above and set FORCE_SELECTED_COL.")
else:
    print("\n✅ Selected column for text modeling:", selected_col)

# Build mdna_series & NLP scores if column exists
if selected_col:
    mdna_series = df[selected_col].fillna('').astype(str).reset_index(drop=True)
    nlp_scores = compute_simple_nlp_scores(mdna_series)

    n = len(mdna_series)
    non_empty_mask = mdna_series.map(lambda s: bool(s.strip()))
    n_nonempty = non_empty_mask.sum()
    max_features = cfg.get('nlp', {}).get('max_features', 5000)
    n_svd = cfg.get('nlp', {}).get('svd_components', 200)

    tfidf_path = os.path.join(models_dir, 'tfidf_vectorizer.joblib')
    svd_path = os.path.join(models_dir, 'tfidf_svd.joblib')

    try:
        print(f"\nRunning TF-IDF on {n_nonempty} documents...")
        vec, X_tfidf = compute_tfidf(mdna_series.tolist(), max_features=max_features, ngram_range=(1,2))
        save_vectorizer(vec, tfidf_path)
        svd = TruncatedSVD(n_components=min(n_svd, X_tfidf.shape[1] - 1), random_state=42)
        X_svd = svd.fit_transform(X_tfidf)
        joblib.dump(svd, svd_path)
        used_tfidf = True
    except Exception as e:
        print("⚠ TF-IDF failed:", e)
        X_svd = np.zeros((n, 1))
        used_tfidf = False

    # Build feature sets
    X_base_num = X_base.select_dtypes(include=[np.number]).reset_index(drop=True)
    tabular_arr = X_base_num.values
    tabular_nlp_df = pd.concat([X_base_num, nlp_scores.reset_index(drop=True)], axis=1).select_dtypes(include=[np.number])
    tabular_nlp_arr = tabular_nlp_df.values
    X_full_dense = np.hstack([tabular_nlp_arr, X_svd])

    feature_sets = {
        'tabular': (tabular_arr, list(X_base_num.columns)),
        'tabular_nlp': (tabular_nlp_arr, list(tabular_nlp_df.columns)),
        'tabular_fulltext': (X_full_dense, list(tabular_nlp_df.columns) + [f"svd_{i}" for i in range(X_svd.shape[1])])
    }

    print("\n✅ Final feature set shapes:")
    print("Tabular:", feature_sets['tabular'][0].shape)
    print("Tabular + NLP numeric:", feature_sets['tabular_nlp'][0].shape)
    print("Full text (with TF-IDF/SVD):", feature_sets['tabular_fulltext'][0].shape)


Candidate text-like columns (sorted by avg token count):


,col,non_empty,avg_len,median_len,avg_tok,ws_ratio,punct_ratio,distinct_ratio
0,company_size,35098,5.788449,5.0,1.114223,0.011422,0.0,0.000171
1,adsh,35098,20.000000,20.0,1.000000,0.000000,0.0,1.000000
2,company_name,35098,12.874437,13.0,1.000000,0.000000,0.0,1.000000
3,sector,35098,8.852413,9.0,1.000000,0.000000,0.0,0.000199
4,rating,35098,2.084706,2.0,1.000000,0.000000,0.0,0.000142



Showing up to 3 sample values for top text candidates:

--- company_size  | non_empty=35098, avg_tok=1.1 ---
  sample 1: Medium
  sample 2: Medium
  sample 3: Medium

--- adsh  | non_empty=35098, avg_tok=1.0 ---
  sample 1: 0000002178-23-000038
  sample 2: 0000002178-23-000082
  sample 3: 0000002178-24-000035

--- company_name  | non_empty=35098, avg_tok=1.0 ---
  sample 1: Company_5
  sample 2: Company_7
  sample 3: Company_9

--- sector  | non_empty=35098, avg_tok=1.0 ---
  sample 1: Consumer
  sample 2: Utilities
  sample 3: Financial

--- rating  | non_empty=35098, avg_tok=1.0 ---
  sample 1: BBB
  sample 2: BB
  sample 3: BBB


⚠ No strong text column found automatically — pick one manually from above and set FORCE_SELECTED_COL.


# ===== Robust training loop cell (run this now) =====

In [19]:
# ===== Robust training loop cell (run this now) =====
import os
import numpy as np
import pandas as pd
import joblib
from pathlib import Path
from src.model_training import ModelTrainer

# Ensure prerequisites
assert 'feature_sets' in globals(), "feature_sets not found — run the feature-building cell first."
assert 'df' in globals() and 'y_binary' in globals() and 'y_multi' in globals(), "df/y_binary/y_multi missing — run data prep cell."

# Paths & trainer
artifacts_dir = cfg['paths']['artifacts']
models_dir = cfg['paths']['models']
os.makedirs(artifacts_dir, exist_ok=True)
os.makedirs(models_dir, exist_ok=True)
trainer = ModelTrainer(cfg, artifacts_dir)

# Load or build train/test indices for reproducibility
def load_or_build_indices():
    n = len(df)
    idx_files = {
        'idx_train_b': os.path.join(models_dir, 'idx_train_b.npy'),
        'idx_test_b': os.path.join(models_dir, 'idx_test_b.npy'),
        'idx_train_m': os.path.join(models_dir, 'idx_train_m.npy'),
        'idx_test_m': os.path.join(models_dir, 'idx_test_m.npy'),
    }
    if all(os.path.exists(p) for p in idx_files.values()):
        idx_train_b = np.load(idx_files['idx_train_b'])
        idx_test_b = np.load(idx_files['idx_test_b'])
        idx_train_m = np.load(idx_files['idx_train_m'])
        idx_test_m = np.load(idx_files['idx_test_m'])
        print("Loaded saved train/test index arrays from models_dir.")
    else:
        print("Saved index arrays not found — generating stratified splits now.")
        from sklearn.model_selection import train_test_split
        indices = np.arange(n)
        test_size = cfg.get('split', {}).get('test_size', 0.2)
        random_state = cfg.get('split', {}).get('random_state', 42)
        idx_train_b, idx_test_b = train_test_split(indices, test_size=test_size, stratify=y_binary, random_state=random_state)[:2]
        idx_train_m, idx_test_m = train_test_split(indices, test_size=test_size, stratify=y_multi, random_state=random_state)[:2]
        # save them
        np.save(idx_files['idx_train_b'], np.sort(idx_train_b))
        np.save(idx_files['idx_test_b'], np.sort(idx_test_b))
        np.save(idx_files['idx_train_m'], np.sort(idx_train_m))
        np.save(idx_files['idx_test_m'], np.sort(idx_test_m))
        print("Saved new index arrays to models_dir.")
    return (np.sort(idx_train_b), np.sort(idx_test_b), np.sort(idx_train_m), np.sort(idx_test_m))

idx_train_b, idx_test_b, idx_train_m, idx_test_m = load_or_build_indices()

# Helper to extract train/test matrices from feature_sets entries (which are tuples (array, feat_names))
def get_train_test_from_feature(feature_key):
    arr, feat_names = feature_sets[feature_key]
    # arr is (n, p)
    X_tr_b = arr[idx_train_b]
    X_te_b = arr[idx_test_b]
    X_tr_m = arr[idx_train_m]
    X_te_m = arr[idx_test_m]
    return X_tr_b, X_te_b, X_tr_m, X_te_m, feat_names

# Which feature sets are available? skip ones that are degenerate (e.g., fulltext all-zero)
available_keys = []
for k in feature_sets.keys():
    arr, names = feature_sets[k]
    # skip if arr has only one column and is all zeros
    if arr.shape[1] == 1 and np.allclose(arr[:,0], 0):
        print(f"Skipping feature set '{k}' (degenerate single-zero column).")
        continue
    available_keys.append(k)
print("Feature sets to run:", available_keys)

# Training loop
summary_rows = []
for feature_key in available_keys:
    print("\n=== Training on feature set:", feature_key, "===")
    X_tr_b, X_te_b, X_tr_m, X_te_m, feat_names = get_train_test_from_feature(feature_key)

    # Binary classification
    print(f"-> Binary task (investment-grade) with {X_tr_b.shape[0]} train rows, {X_te_b.shape[0]} test rows")
    res_bin = trainer.run(X_tr_b, X_te_b, y_binary[idx_train_b], y_binary[idx_test_b], feature_set=feature_key, task_name='binary')
    for mdl, info in res_bin.items():
        summary_rows.append({
            'feature_set': feature_key,
            'task': 'binary',
            'model': mdl,
            'accuracy': info['metrics']['accuracy'],
            'model_path': info['model_path']
        })

    # Multi-class classification
    print(f"-> Multi-class task (rating) with {X_tr_m.shape[0]} train rows, {X_te_m.shape[0]} test rows")
    res_multi = trainer.run(X_tr_m, X_te_m, y_multi[idx_train_m], y_multi[idx_test_m], feature_set=feature_key, task_name='multi')
    for mdl, info in res_multi.items():
        summary_rows.append({
            'feature_set': feature_key,
            'task': 'multi',
            'model': mdl,
            'accuracy': info['metrics']['accuracy'],
            'model_path': info['model_path']
        })

# Save summary DataFrame
summary_df = pd.DataFrame(summary_rows)
summary_path = os.path.join(artifacts_dir, 'training_summary.csv')
summary_df.to_csv(summary_path, index=False)
joblib.dump(summary_df, os.path.join(artifacts_dir, 'training_summary.joblib'))
print("\nTraining completed. Summary saved to:", summary_path)

# Show summary and best models per (feature_set, task)
display(summary_df.sort_values(['feature_set','task','model']).reset_index(drop=True))
best_df = summary_df.loc[summary_df.groupby(['feature_set','task'])['accuracy'].idxmax()].reset_index(drop=True)
print("\nBest models by feature set and task:")
display(best_df)


Loaded saved train/test index arrays from models_dir.
Feature sets to run: ['tabular', 'tabular_nlp', 'tabular_fulltext']

=== Training on feature set: tabular ===
-> Binary task (investment-grade) with 28078 train rows, 7020 test rows
-> Multi-class task (rating) with 28078 train rows, 7020 test rows

=== Training on feature set: tabular_nlp ===
-> Binary task (investment-grade) with 28078 train rows, 7020 test rows
-> Multi-class task (rating) with 28078 train rows, 7020 test rows

=== Training on feature set: tabular_fulltext ===
-> Binary task (investment-grade) with 28078 train rows, 7020 test rows
-> Multi-class task (rating) with 28078 train rows, 7020 test rows

Training completed. Summary saved to: data/processed/model_artifacts\training_summary.csv


,feature_set,task,model,accuracy,model_path
0,tabular,binary,gradient_boosting,1.000000,data/processed/model_artifacts\tabular__binary...
1,tabular,binary,logistic_regression,1.000000,data/processed/model_artifacts\tabular__binary...
2,tabular,binary,random_forest,1.000000,data/processed/model_artifacts\tabular__binary...
3,tabular,binary,svm,0.998860,data/processed/model_artifacts\tabular__binary...
4,tabular,multi,gradient_boosting,0.959402,data/processed/model_artifacts\tabular__multi_...
5,tabular,multi,logistic_regression,0.696581,data/processed/model_artifacts\tabular__multi_...
6,tabular,multi,random_forest,0.949430,data/processed/model_artifacts\tabular__multi_...
7,tabular,multi,svm,0.712251,data/processed/model_artifacts\tabular__multi_...
8,tabular_fulltext,binary,gradient_boosting,1.000000,data/processed/model_artifacts\tabular_fulltex...
9,tabular_fulltext,binary,logistic_regression,1.000000,data/processed/model_artifacts\tabular_fulltex...



Best models by feature set and task:


,feature_set,task,model,accuracy,model_path
0,tabular,binary,random_forest,1.000000,data/processed/model_artifacts\tabular__binary...
1,tabular,multi,gradient_boosting,0.959402,data/processed/model_artifacts\tabular__multi_...
2,tabular_fulltext,binary,random_forest,1.000000,data/processed/model_artifacts\tabular_fulltex...
3,tabular_fulltext,multi,gradient_boosting,0.960256,data/processed/model_artifacts\tabular_fulltex...
4,tabular_nlp,binary,random_forest,1.000000,data/processed/model_artifacts\tabular_nlp__bi...
5,tabular_nlp,multi,gradient_boosting,0.959402,data/processed/model_artifacts\tabular_nlp__mu...
